In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from imblearn.over_sampling import SMOTE
import joblib


In [2]:
df = pd.read_csv("../../data/raw/heart.csv")

print(df.head())
print(df.info())
print(df["HeartDisease"].value_counts())

   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age    

In [3]:
df = pd.get_dummies(df, drop_first=True)

print(df.isnull().sum())
df = df.dropna()

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Age                  0
RestingBP            0
Cholesterol          0
FastingBS            0
MaxHR                0
Oldpeak              0
HeartDisease         0
Sex_M                0
ChestPainType_ATA    0
ChestPainType_NAP    0
ChestPainType_TA     0
RestingECG_Normal    0
RestingECG_ST        0
ExerciseAngina_Y     0
ST_Slope_Flat        0
ST_Slope_Up          0
dtype: int64
Features shape: (918, 15)
Target shape: (918,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
scaler_base = StandardScaler()
X_train_scaled = scaler_base.fit_transform(X_train)
X_test_scaled = scaler_base.transform(X_test)

In [6]:
def evaluate(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)
    print("\nClassification Report:\n", classification_report(y_eval, y_pred, zero_division=0))

    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(probability=True),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []

for name, model in models.items():
    print(f"\n===== {name} =====")
    if name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_scaled, y_train)
        metrics = evaluate(model, X_test_scaled, y_test)
    else:
        model.fit(X_train, y_train)
        metrics = evaluate(model, X_test, y_test)

    results.append({"Model": name, **metrics})

results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)
print("\n=== FINAL COMPARISON ===")
print(results_df)


===== Logistic Regression =====
Accuracy: 0.8913043478260869
Precision: 0.8796296296296297
Recall: 0.9313725490196079
F1 Score: 0.9047619047619048

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.84      0.87        82
           1       0.88      0.93      0.90       102

    accuracy                           0.89       184
   macro avg       0.89      0.89      0.89       184
weighted avg       0.89      0.89      0.89       184


===== Decision Tree =====
Accuracy: 0.7989130434782609
Precision: 0.8155339805825242
Recall: 0.8235294117647058
F1 Score: 0.8195121951219512

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.77      0.77        82
           1       0.82      0.82      0.82       102

    accuracy                           0.80       184
   macro avg       0.80      0.80      0.80       184
weighted avg       0.80      0.80      0.80       184


=====

In [8]:
top_models = results_df.head(2)["Model"].values
print("Top Models:", top_models)

Top Models: ['SVM' 'Logistic Regression']


In [9]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_res.value_counts())

Before SMOTE:
 HeartDisease
1    406
0    328
Name: count, dtype: int64
After SMOTE:
 HeartDisease
1    406
0    406
Name: count, dtype: int64


In [10]:

scaler = StandardScaler()

X_train_res_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_res),
    columns=X_train.columns,
    index=X_train_res.index
)

X_test_scaled_smote = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)


In [11]:
final_results = {}

for name in top_models:
    model = models[name]
    print(f"\n===== {name} (After SMOTE) =====")

    if name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_res_scaled, y_train_res)
        probs = model.predict_proba(X_test_scaled_smote)[:, 1]
    else:
        model.fit(X_train_res, y_train_res)
        probs = model.predict_proba(X_test)[:, 1]

    best_thresh = 0.5
    best_f1 = -1

    for t in np.arange(0.1, 0.9, 0.05):
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_test, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = float(t)

    final_preds = (probs >= best_thresh).astype(int)

    metrics = {
        "Accuracy": accuracy_score(y_test, final_preds),
        "Precision": precision_score(y_test, final_preds, zero_division=0),
        "Recall": recall_score(y_test, final_preds, zero_division=0),
        "F1": f1_score(y_test, final_preds, zero_division=0),
        "Threshold": best_thresh
    }

    print(f"Best Threshold: {best_thresh:.2f} | Best F1: {best_f1:.3f}")
    print(metrics)

    final_results[name] = {"model": model, **metrics}


===== SVM (After SMOTE) =====
Best Threshold: 0.35 | Best F1: 0.912
{'Accuracy': 0.8967391304347826, 'Precision': 0.8672566371681416, 'Recall': 0.9607843137254902, 'F1': 0.9116279069767442, 'Threshold': 0.3500000000000001}

===== Logistic Regression (After SMOTE) =====
Best Threshold: 0.55 | Best F1: 0.903
{'Accuracy': 0.8913043478260869, 'Precision': 0.8942307692307693, 'Recall': 0.9117647058823529, 'F1': 0.9029126213592233, 'Threshold': 0.5500000000000002}


In [12]:
best_model_name = max(final_results, key=lambda x: final_results[x]["F1"])
best_model = final_results[best_model_name]["model"]
best_threshold = final_results[best_model_name]["Threshold"]

print("\nFINAL MODEL:", best_model_name)
print("Threshold:", best_threshold)


FINAL MODEL: SVM
Threshold: 0.3500000000000001


In [13]:
out_dir = Path("../../models/heart")
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, out_dir / "model.pkl")
joblib.dump(scaler, out_dir / "scaler.pkl")
joblib.dump(best_threshold, out_dir / "threshold.pkl")

print("Heart model artifacts saved to:", out_dir)

Heart model artifacts saved to: ..\..\models\heart


In [14]:
sample = X_test.iloc[0:1]

if best_model_name in ["Logistic Regression", "SVM"]:
    sample_scaled = pd.DataFrame(
        scaler.transform(sample),
        columns=X_test.columns,
        index=sample.index
    )
    prob = best_model.predict_proba(sample_scaled)[0][1]
else:
    prob = best_model.predict_proba(sample)[0][1]

pred = 1 if prob >= best_threshold else 0   # <- this was missing

print("Probability:", prob)
print("Prediction:", pred)
print("Actual:", y_test.iloc[0])


Probability: 0.9432070280397767
Prediction: 1
Actual: 1


In [16]:
import random

idx = random.randint(0, len(X) - 1)

row = X.iloc[idx]
actual = y.iloc[idx]

# For scaled models (LR/SVM), use scaler
if best_model_name in ["Logistic Regression", "SVM"]:
    row_scaled = pd.DataFrame(
        scaler.transform(row.to_frame().T),
        columns=X.columns,
        index=[idx]
    )
    prob = best_model.predict_proba(row_scaled)[0][1]
else:
    prob = best_model.predict_proba(row.to_frame().T)[0][1]

pred = 1 if prob >= best_threshold else 0

print("INDEX:", idx)
print("MODEL INPUT (one-hot):")
print(row)

print("ACTUAL:", actual)
print("PREDICTED:", pred)
print("PROB:", prob)


INDEX: 528
MODEL INPUT (one-hot):
Age                     49
RestingBP              131
Cholesterol            142
FastingBS                0
MaxHR                  127
Oldpeak                1.5
Sex_M                 True
ChestPainType_ATA    False
ChestPainType_NAP     True
ChestPainType_TA     False
RestingECG_Normal     True
RestingECG_ST        False
ExerciseAngina_Y      True
ST_Slope_Flat         True
ST_Slope_Up          False
Name: 528, dtype: object
ACTUAL: 1
PREDICTED: 1
PROB: 0.8836357932415182
